In [1]:
#| default_exp lawa

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [4]:
#| export
from os import getenv
model_path = getenv("MODEL")
from rest.gen import generate


In [5]:
model_path = 'lawa'

In [6]:
#| export
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast
full_path = f'./models/{model_path}'
tokenizer = PreTrainedTokenizerFast.from_pretrained(full_path)
model = AutoModelForCausalLM.from_pretrained(
    full_path,
    #device_map="auto",
    torch_dtype='auto'
).eval()


In [7]:
#| export
import torch

seq_length = 1024
device='cuda'
model.to(device);
model.eval();

In [8]:
tokenizer.encode('На словах ты Лев Толстой, а на деле')

[128000,
 85587,
 92207,
 100676,
 103617,
 100562,
 33742,
 51418,
 7975,
 6735,
 16742,
 11,
 21022,
 13373,
 120664]

In [9]:
sum(p.numel() for p in model.parameters())

1235814400

In [10]:
#| export
from front.common import process_seq

def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool, temperature:float=0.5):
    lm_text = prompt
    input_ids=torch.tensor([tokenizer.encode(lm_text)]).cuda()
    output_ids = model.generate(input_ids, do_sample=True, temperature=temperature, repetition_penalty=5.0, typical_p=0.9, top_k=10, top_p=0.95, #watermark=False,
                        max_new_tokens=length, 
                        num_return_sequences=num_samples,)
    input_len = len(input_ids[0])
    output_ids = output_ids[:,input_len-1:]
    result = [tokenizer.decode(o[1:]).replace('\n', ' ') for o in output_ids]
    result = process_seq(result)
    return result


In [11]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


CPU times: user 920 ms, sys: 94.5 ms, total: 1.01 s
Wall time: 1.01 s


[' Коля Мурзенко. —\xa0 Какая разница?',
 ' Михаил Юльевич Соловьёв. Это я понял после того как вы два раза меня спросили «кто такой»?',
 ' Сорос. – Почему?',
 ' Сорокин. —\xa0 Это я не знаю даже… Я только слышал это выражение много раз и думаю – что же такое? Ведь по-английски «pornography»[ 2 - Пиротехника.']